In [ ]:
!pip install matplotlib
!pip install plotly

In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import plotly.express as px

In [ ]:
comments = pd.read_csv(r"C:\Users\rickymht\Documents\SQL practice\Python project\Youtube Case study\UScomments.csv", on_bad_lines = 'skip')

In [ ]:
comments

In [ ]:
comments.duplicated()

In [ ]:
#To filter out duplicated rows from the dataset and sorts the values on the basis of text in comment_text
comments[comments.duplicated(keep = False)].sort_values('comment_text')

In [ ]:
#To drop duplicate rows from the data and save the changes to dataset
comments = comments.drop_duplicates()


In [ ]:
#To check the null values in the dataset
comments.isnull()

In [ ]:
#As the data is not clear we will sum the number of null values for each variable
comments.isnull().sum()

In [ ]:
comments.dropna(inplace = True)

In [ ]:
#The code shows we have dropped the rows where the value was null
comments.isnull().sum()

### 2. Sentiment Analysis

In [ ]:
!pip install nltk

In [ ]:
import nltk

In [ ]:
#lexicon is a built in list of words with sentiment scores
nltk.download('vader_lexicon')

In [ ]:
'''Calling that specific import line tells Python to look inside the nltk library, go into its sentiment folder, find a specific tool called 
SentimentIntensityAnalyzer, and bring it into notebook so it can be used.'''
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
sia = SentimentIntensityAnalyzer()

In [ ]:
comments['comment_text']

In [ ]:
#defining a function to analyse the sentiments from comment
def analyze_comment_sentiment(comment):
    compound = sia.polarity_scores(comment)['compound']

    if compound > 0.05:
        label = 'Positive'
        insight = 'The comment expresses positive emotion or approval'

    elif compound < -0.05:
        label = 'Negative'
        insight = 'The comment shows dis-satisfaction or negative emotion'

    else:
        label = 'Neutral'
        insight = 'The comment is emotionally neutral'

    return{
        'label' : label,
        'score' : compound,
        'insight' : insight }
        

In [ ]:
analyze_comment_sentiment("Worst edited video ever seen")

### 3. Emoji's Analysis

We have around 400k text comment data and to simplfy our analysis we will extract only the emoji from our comments and try to understand the viewer response

In [ ]:
!pip install emoji


In [ ]:
import emoji

In [ ]:
# For reference to get the first 25 comments from the data 
comments['comment_text'].head(25)

In [ ]:
# To check the number of different emojis in the library
total_emojis = len(emoji.EMOJI_DATA)
print(f'My Python library knows about {total_emojis} different emojis')

In [ ]:
# trying to extract the emoji from a text
comment = 'trending 😉'

emoji_list = [] #a blank list that will store the lists of all the emojis  extracted from comments

for char in comment:
    if char in emoji.EMOJI_DATA:
        emoji_list.append(char)

In [ ]:
emoji_list

In [ ]:
# to extract data from comments dataset

all_emoji = []

for comment in comments['comment_text'].dropna(): # to extract the comments and drop any missing values
    for char in comment:
        if char in emoji.EMOJI_DATA:
            all_emoji.append(char)
all_emoji[0:10] #to check only the first 10 values

In [ ]:
len(all_emoji) #number of different emojis in the extracted data

In [ ]:
from collections import Counter

In [ ]:
emoji_count = Counter(all_emoji).most_common(10)
emoji_count

In [ ]:
emojis = [emoji for emoji, count in emoji_count] #it gives out the list of top 10 emoji
emojis

In [ ]:
count = [count for emoji, count in emoji_count]
count

In [ ]:
# bar plot to visualize the count per emoji
px.bar(x = emojis, y = count, title = 'Most used emojis in comment', 
       labels = {'x' : 'Emojis',
            'y': 'Count'})

### 4. Collect entire data

In [ ]:
import os

In [ ]:
files = os.listdir(r'C:\Users\rickymht\Documents\SQL practice\Python project\Youtube Case study\additional_data')

In [ ]:
#Other country data
files

In [ ]:
files_csv = [file for file in files if '.csv' in file]

In [ ]:
files_csv

In [ ]:
full_df = pd.DataFrame()

path = r'C:\Users\rickymht\Documents\SQL practice\Python project\Youtube Case study\additional_data'

for file in files_csv:
    current_df = pd.read_csv(path+'/'+file, encoding = 'iso=8859-1', on_bad_lines = 'skip')
    full_df = pd.concat([current_df, full_df], ignore_index = True)

In [ ]:
full_df.shape

In [ ]:
# to view the top 3 rows from the dataframe
full_df.head(3)

In [ ]:
# to view all the columns from the dataframe
full_df.columns

### 5. How to export your data into (CSV, JSON, DB)

In [ ]:
# wherever we have boolean false, the row in not duplicate
full_df.duplicated()

In [ ]:
full_df[full_df.duplicated()].shape

In [ ]:
full_df = full_df.drop_duplicates()

In [ ]:
full_df.shape

In [ ]:
# to export data in CSV
full_df[0:5000].to_csv(r'C:\Users\rickymht\Documents\SQL practice\Python project\Export data\youtube_data.csv', index = False)

### 6. Which category dominates Youtube ?

In [ ]:
full_df.dtypes
#trending_date data type should be datae time rather than a object (which is incorrect)

In [ ]:
full_df['trending_date']

In [ ]:
full_df['trending_date'] = pd.to_datetime(full_df['trending_date'], format = '%y.%d.%m')

In [ ]:
full_df.dtypes
#the valued of trending date has changed to date time now

In the data type we see we do not have any category names column but only a category ID. To get the names of the category we will import one of the files which has the category code as well as the category name

In [ ]:
import json

In [ ]:
path = r'C:\Users\rickymht\Documents\SQL practice\Python project\Youtube Case study\additional_data\US_category_id.json'

In [ ]:
with open(path, 'r', encoding = 'utf-8') as f:
    data = json.load(f)

In [ ]:
data

In [ ]:
data['items'][0]

In [ ]:
# to fetch the category name
data['items'][0]['snippet']['title']

In [ ]:
# to create a category dictionary
cat_dict = {}

for item in data['items']:
    cat_dict[int(item['id'])] = item['snippet']['title']

In [ ]:
cat_dict

In [ ]:
full_df['category_id']

In [ ]:
#mapping the category id, with the category dictionary 
full_df['category_name'] = full_df['category_id'].map(cat_dict)

In [ ]:
full_df.head(3)

In [ ]:
#to pivot the data on views for each category for the particular data 
pivot_df = full_df.groupby(['trending_date', 'category_name'])['views'].sum().unstack(fill_value = 0)
pivot_df

In [ ]:
area_chart = px.area(
    data_frame = pivot_df, 
    x = pivot_df.index, 
    y = pivot_df.columns, 
    title = 'Trending momentum over time by category'
)

In [ ]:
area_chart

In [ ]:
# as there are multiple categories stacked in the graph we will filter out the first top 6 categories 

top_categories = full_df.groupby(['category_name'])['views'].sum().nlargest(6).index
top_categories

In [ ]:
filtered_df = pivot_df[top_categories]
filtered_df

In [ ]:
area_chart = px.area(
    data_frame = pivot_df, 
    x = filtered_df.index, 
    y = filtered_df.columns, 
    title = 'Trending momentum over time by category'
)

In [ ]:
area_chart

### 7. Do viral videos actually get engagement ?


In [ ]:
full_df.columns
#we don't have a column for engagment rate

In [ ]:
#engagement rate = [Interaction (comments + likes)]/views
full_df['engagement_rate'] = (full_df['likes'] + full_df['comment_count']) / full_df['views']

In [ ]:
bubble_plot = px.scatter(
    full_df, 
    x = 'views', 
    y = 'engagement_rate',
    size = 'comment_count', 
    color = 'category_name',
    hover_name = 'title',
    title = 'Engagement bubble map: Views Vs Engagement rate',
    size_max = 60
)

In [ ]:
bubble_plot

In [ ]:
bubble_plot.update_xaxes(type = 'log')

### 8. Views Vs Engagement: Inside Youtube's algorithim

In [ ]:
full_df.columns

In [ ]:
category_metrics = full_df.groupby('category_name').agg(
                                    total_views = ('views', 'sum'),
                                    avg_engagement_efficiency = ('engagement_rate', 'mean'),
                                    video_count = ('video_id', 'count')).reset_index()

In [ ]:
category_metrics

In [ ]:
treemap = px.treemap(
    category_metrics, 
    path = ['category_name'],
    values = 'total_views',
    color = 'avg_engagement_efficiency',
    color_continuous_scale = 'RdYlGn',
    title = 'Category attention share with engagement efficiency overlay', 
    hover_data = {
        'total_views': ':, .0f',
        'avg_engagement_efficiency': ':, .3f',
        'video_count': True
    }
)


In [ ]:
treemap

### 9. Is the audience actually engaged ?


In [ ]:
full_df.columns

In [ ]:
full_df['engagement_rate'].describe()

In [ ]:
category_engagement_stats = full_df.groupby('category_name')['engagement_rate'].describe()
category_engagement_stats.sort_values('mean', ascending = False)

In [ ]:
box = px.box(full_df, 
       x = 'category_name',
       y = 'engagement_rate',
       color = 'category_name',
       title = 'Audience engagement by category')

In [ ]:
box